# Verificar la carga del checkpoint publicado — issue #4

Sesión con **GPU T4 x2**, Internet **on**, Secrets: `HF_TOKEN` (alcanza con lectura). Hace lo mismo que la cátedra: `entrega.construir_agente()`
con `ADMISION_MODELO_REPO` / `ADMISION_MODELO_REVISION`, cuenta que el modelo se cargue **una sola vez** y que la primera respuesta sea
el JSON de `leer_triage`. Sin reentrenar. Dura unos 10–15 minutos.

In [ ]:
RAMA = "juli/entrenamiento"
REPO_HF = "genai-grupo-2/emergency-admission-agent"
REVISION = "292c8137984283d426de735309d108459b5e51f9"

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["PYTHONUTF8"] = "1"
os.environ["ADMISION_DEVICE_MAP"] = "auto"   # reparte el modelo entre las 2 T4
REPO = "/tmp/repo"

def sh(comando):
    print("$", comando, flush=True)
    subprocess.run(comando, shell=True, check=True, cwd=REPO if os.path.isdir(REPO) else None)

if not os.path.isdir(REPO):
    sh(f"git clone -q -b {RAMA} https://github.com/genai-grupo-2/emergency-admission-agent.git {REPO}")
else:
    sh(f"git fetch -q origin {RAMA} && git checkout -q {RAMA} && git reset -q --hard origin/{RAMA}")
sh("git log --oneline -1")

In [ ]:
# Mismas versiones que la imagen de modal_app.py, instaladas en el .venv del repo (--python: Kaggle configura uv
# para instalar en el Python del sistema). Si la rueda de torch por defecto no ve las GPU (driver viejo), se prueba
# la que elige uv según el driver y después las de otras versiones de CUDA. La que quedó se anota en corridas/entornos/.
sh("pip install -q uv")
sh("uv sync --locked -q")
PIP = "uv pip install -q --python .venv/bin/python"
sh(f"{PIP} torch==2.14.1 transformers==5.20.2 accelerate==1.15.0")
ver = (".venv/bin/python -c \"import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), "
       "torch.cuda.device_count()); assert torch.cuda.device_count() >= 2\"")
intentos = [None, "--torch-backend=auto"] + [f"--index-url https://download.pytorch.org/whl/{c}"
                                           for c in ["cu130", "cu129", "cu128", "cu126"]]
for intento in intentos:
    if intento and subprocess.run(f"{PIP} --reinstall torch==2.14.1 {intento}", shell=True, cwd=REPO).returncode:
        print("no hay torch 2.14.1 con", intento)
        continue
    if subprocess.run(ver, shell=True, cwd=REPO).returncode == 0:
        print("torch listo con", intento or "la rueda por defecto de PyPI")
        break
else:
    raise RuntimeError("torch 2.14.1 no ve las dos GPU con ninguna rueda: pasar la salida de nvidia-smi y de esta celda")

In [ ]:
sh(f".venv/bin/python -m entrenar.verificar_carga --repo {REPO_HF} --revision {REVISION}")